In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import GammaRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score

In [2]:
# Load cleaned data
df = pd.read_csv('./Cleaned Data/cleaned_data.csv')

In [3]:
# Keep only positive targets to avoid issues with log transformation so the model won't encounter zero or negative values
df = df[df['Total Time Spent'] > 0].reset_index(drop=True)

### Define Features and Target Variable

In [4]:
# define features and target variables
text_features = 'Summary'
categorical_features = ['Issue Type', 'Priority']
target = 'Total Time Spent'

### Prepare Feature Matrix and Target Vector

In [5]:
X = df[[text_features] + categorical_features]
y = df[target]

### Log Transformation of Target Variable

Log transformation is an effective technique to help prevent a regression model from predicting negative numbers for a target variable that is naturally positive. By modeling the logarithm of the target variable, the predictions are constrained to be non-negative when transformed back to the original scale.

References:

https://medium.com/@stevechesa/log-transforming-target-variables-and-enhancing-tree-ensembles-53be435d8041

https://stackoverflow.com/questions/66334730/ways-to-handle-negative-value-of-prediction-in-regression-model

In [6]:
# log-transform the target variable to prevent the model from predicting negative values
y_log = np.log1p(y)

### Create Preprocessing Pipeline
Pipline that processes both text and categorical features into a format suitable for modeling.

In [7]:
# prepare feature matrix X and target vector y
final_preprocessor = ColumnTransformer(
    transformers=[
        (
            'text',
            TfidfVectorizer(
                max_features=300,
                min_df=3,
                ngram_range=(1, 2)
            ),
            'Summary'
        ),
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore'),
            categorical_features
        )
    ]
)

### Hyperparameter Tuning with Grid Search
This uses GridSearchCV to find the best hyperparameters for the GammaRegressor model. Searches over a range of alpha values and maximum iterations to optimize model performance based on cross-validated mean absolute error.

In [8]:
from sklearn.model_selection import GridSearchCV

# use hyperparameter tuning to find the best parameters for the model

param_grid = {
    'regressor__alpha': np.logspace(-4, 2, 20),
    'regressor__max_iter': [1000, 2000, 3000]
}
model_for_tuning = Pipeline(
    steps=[
        ('preprocessor', final_preprocessor),
        ('regressor', GammaRegressor())
    ]
)
grid_search = GridSearchCV(
    model_for_tuning,
    param_grid,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

grid_search.fit(X, y_log)
print("Best parameters found: ", grid_search.best_params_['regressor__alpha'], grid_search.best_params_['regressor__max_iter'])
print("Best CV MAE: ", -grid_search.best_score_)

Best parameters found:  0.0001 1000
Best CV MAE:  0.6635295976933779


### Final Model Training and Evaluation
This section creates the final model pipeline using the best hyperparameters found during grid search. It evaluates the model using K-Fold cross-validation to provide a robust estimate of performance.

In [9]:
# create the final model pipeline
model = Pipeline(
    steps=[
        ('preprocessor', final_preprocessor),
        ('regressor', GammaRegressor(
            alpha=grid_search.best_params_['regressor__alpha'],
            max_iter=grid_search.best_params_['regressor__max_iter']
        ))
    ]
)


### K-Fold Cross-Validation
 K-Fold cross-validation is used to evaluate the model's performance more robustly by splitting the data into multiple folds and averaging the results. This helps to ensure that the model's performance is not overly dependent on a particular train-test split.

In [10]:
# use K-Fold cross-validation to evaluate the model
# this provides a more robust estimate of model performance
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    model,
    X,
    y_log,
    cv=kf,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

mae_scores = -cv_scores

print("K-Fold MAE scores:", mae_scores)
print(f"Average MAE: {mae_scores.mean():.2f}")
print(f"Std Dev: {mae_scores.std():.2f}")

K-Fold MAE scores: [0.66933471 0.68107608 0.67594152 0.67474294 0.69878979]
Average MAE: 0.68
Std Dev: 0.01


K-Fold Results:
K-Fold MAE scores: [0.66956724 0.68329858 0.67322059 0.6755314  0.69841089]
Average MAE: 0.68
Std Dev: 0.01

Spread: There isn’t a large variation between folds which indicates the model’s performance is relatively consistent across different subsets of the data.

Standard Deviation: The low standard deviation (0.01) suggests that the model's performance is stable and not overly sensitive to the specific data points in each fold.



### Train-Test Split and Final Evaluation
This section splits the data into training and testing sets, fits the model on the training data, and evaluates it on the test data. The results, including actual and predicted values, are compiled into a DataFrame for further analysis.

Test Size: 25%

Train Size: 75%

Random State: 42

In [11]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

### Fit Model on Training Data
Log-transform the target variable before fitting the model to ensure predictions remain positive.

In [12]:
# Log transform target
y_train_log = np.log1p(y_train)

In [13]:
# Fit model
model.fit(X_train, y_train_log)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('text', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### Make Predictions and Evaluate
The model makes predictions on the test set, which are then exponentiated back to the original scale

In [14]:
# Predict
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)

In [15]:
results = pd.DataFrame({
    'Summary': df.loc[X_test.index, 'Summary'],
    'Issue Type': df.loc[X_test.index, 'Issue Type'],
    'Priority': df.loc[X_test.index, 'Priority'],
    'Original Estimated Time': df.loc[X_test.index, 'Original Time Estimated'],
    'Actual Time Spent': y_test,
    'Predicted Time Spent': y_pred
})

In [16]:
results

,Summary,Issue Type,Priority,Original Estimated Time,Actual Time Spent,Predicted Time Spent
5601,"['Add', 'endpoint', 'upserting', 'disciplines']",Subtask,Major,60,60,100.491838
2108,"['Add', 'endpoint', 'saving', 'configuration']",Subtask,Major,120,205,180.480471
2516,"['I', 'get', 'analysis', 'AR', 'I', 'first', '...",Story,Major,0,75,61.356824
7318,['Feedback'],Subtask,Major,0,20,80.950678
33,"['What', 'caused', 'change', 'value', 'invento...",Subtask,Major,0,141,43.318567
...,...,...,...,...,...,...
4269,"['As', 'child', 'I', 'able', 'login', 'code']",Story,Major,0,120,154.871287
6762,"['update', 'PO', 'method', 'approve', 'POs']",Subtask,Major,30,60,75.466036
6965,"['Upgrade', 'version', 'opentelemetry', 'use']",Task,Critical,0,240,72.862687
4184,"['add', 'ela', 'writing', 'subject']",Subtask,Major,120,300,73.133788


In [17]:
results['Absolute Error Predicted'] = np.abs(
    results['Actual Time Spent'] - results['Predicted Time Spent']
)


In [18]:
# absolute error original
results['Absolute Error Original'] = np.abs(
    results['Actual Time Spent'] - results['Original Estimated Time']
)

In [19]:
results

,Summary,Issue Type,Priority,Original Estimated Time,Actual Time Spent,Predicted Time Spent,Absolute Error Predicted,Absolute Error Original
5601,"['Add', 'endpoint', 'upserting', 'disciplines']",Subtask,Major,60,60,100.491838,40.491838,0
2108,"['Add', 'endpoint', 'saving', 'configuration']",Subtask,Major,120,205,180.480471,24.519529,85
2516,"['I', 'get', 'analysis', 'AR', 'I', 'first', '...",Story,Major,0,75,61.356824,13.643176,75
7318,['Feedback'],Subtask,Major,0,20,80.950678,60.950678,20
33,"['What', 'caused', 'change', 'value', 'invento...",Subtask,Major,0,141,43.318567,97.681433,141
...,...,...,...,...,...,...,...,...
4269,"['As', 'child', 'I', 'able', 'login', 'code']",Story,Major,0,120,154.871287,34.871287,120
6762,"['update', 'PO', 'method', 'approve', 'POs']",Subtask,Major,30,60,75.466036,15.466036,30
6965,"['Upgrade', 'version', 'opentelemetry', 'use']",Task,Critical,0,240,72.862687,167.137313,240
4184,"['add', 'ela', 'writing', 'subject']",Subtask,Major,120,300,73.133788,226.866212,180


In [20]:
# mean absolute error original
mae_original = results['Absolute Error Original'].mean()
print(f"Mean Absolute Error (Original): {mae_original:.2f}")

Mean Absolute Error (Original): 90.41


In [21]:
# mean absolute error predicted
mae_predicted = results['Absolute Error Predicted'].mean()
print(f"Mean Absolute Error (Predicted): {mae_predicted:.2f}")

Mean Absolute Error (Predicted): 79.28


In [22]:
results.to_csv('prediction_results.csv', index=False)

In [23]:
# average actual time spent
print(f"Average Actual Time Spent: {results['Actual Time Spent'].mean():.2f}")

Average Actual Time Spent: 132.17


In [24]:
# absolute error original
results['Absolute Error Average'] = np.abs(
    results['Actual Time Spent'] - 132
)

In [25]:
# mean absolute error average
mae_average = results['Absolute Error Average'].mean()
print(f"Mean Absolute Error (Average): {mae_average:.2f}")

Mean Absolute Error (Average): 101.96


In [26]:
results['Actual Time Spent (hrs)'] = results['Actual Time Spent'] / 3600
results['Original Estimated Time (hrs)'] = results['Original Estimated Time'] / 3600
results['Predicted Time Spent (hrs)'] = results['Predicted Time Spent'] / 3600

In [28]:
HOURLY_RATE = 50

# --- Revenue and cost for each scenario ---
results['Budgeted Revenue (Original)'] = results['Original Estimated Time (hrs)'] * HOURLY_RATE
results['Budgeted Revenue (Predicted)'] = results['Predicted Time Spent (hrs)'] * HOURLY_RATE
results['Actual Cost'] = results['Actual Time Spent (hrs)'] * HOURLY_RATE

# --- Profit/Loss per task ---
results['Profit (Original)'] = results['Budgeted Revenue (Original)'] - results['Actual Cost']
results['Profit (Predicted)'] = results['Budgeted Revenue (Predicted)'] - results['Actual Cost']

# --- Summary ---
print("=== Financial Performance Summary ===")
print(f"Total Over-budget tasks (Original): {(results['Profit (Original)'] < 0).sum()}")
print(f"Total Over-budget tasks (Predicted): {(results['Profit (Predicted)'] < 0).sum()}")
print()
total_loss_original = results[results['Profit (Original)'] < 0]['Profit (Original)'].sum()
total_loss_predicted = results[results['Profit (Predicted)'] < 0]['Profit (Predicted)'].sum()
print(f"Total $ Lost to Over-runs (Original):  ${abs(total_loss_original):,.2f}")
print(f"Total $ Lost to Over-runs (Predicted): ${abs(total_loss_predicted):,.2f}")
print(f"\nNet Improvement from Model: ${abs(total_loss_original) - abs(total_loss_predicted):,.2f}")

=== Financial Performance Summary ===
Total Over-budget tasks (Original): 1761
Total Over-budget tasks (Predicted): 1298

Total $ Lost to Over-runs (Original):  $3,023.67
Total $ Lost to Over-runs (Predicted): $2,063.75

Net Improvement from Model: $959.92
